<a href="https://colab.research.google.com/github/samuel12-lab/ai-server/blob/main/Another_copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# =====================================================================
# 1. Core MZI Engine
# =====================================================================
class MZICore:
    """Simulates a Mach-Zehnder Interferometer (MZI) mesh for matrix multiplication."""

    @staticmethod
    def mzi_2x2(theta: float, phi: float) -> np.ndarray:
        """Standard 2x2 optical unitary matrix representation of an MZI."""
        return 0.5 * np.array([
            [np.exp(1j * phi) * (np.exp(1j * theta) - 1), 1j * (np.exp(1j * theta) + 1)],
            [1j * np.exp(1j * phi) * (np.exp(1j * theta) + 1), -(np.exp(1j * theta) - 1)]
        ], dtype=complex)

    @staticmethod
    def simulate_matrix_vector_mult(W: np.ndarray, x: np.ndarray) -> np.ndarray:
        """
        Simulates Optical Matrix-Vector Multiplication (MVM).
        Uses SVD decomposition (U @ S @ V_dagger) as an ideal MZI mesh proxy.
        This reconstructs W @ x exactly (SVD proxy is lossless), so any
        deviation from a linear NumPy baseline must come from encoding
        or readout, not from this stage.
        """
        U, S, Vh = np.linalg.svd(W, full_matrices=False)
        v_out = Vh @ x
        s_out = S * v_out
        u_out = U @ s_out
        return u_out


# =====================================================================
# 2. Input Encoding and Data Preparation
# =====================================================================
class InputEncoder:
    """
    Encodes real-world signals/patches into optical field amplitudes.

    FIX: the original implementation mapped x -> normalized magnitude AND
    a data-dependent phase (normalized * pi). That makes x_encoded a
    genuinely different vector from x (different magnitude *and* a phase
    term), so W @ x_encoded can never equal W @ x. If the goal is for the
    pipeline to validate against a plain linear W @ x baseline, the
    encoding has to be magnitude-only, zero-phase, and not rescale the
    vector: x_encoded = x (as a complex-valued signal with phase 0).
    """

    @staticmethod
    def encode_vector(x: np.ndarray) -> np.ndarray:
        """Amplitude-only encoding: real input becomes a zero-phase optical field."""
        x_flat = np.asarray(x, dtype=float).flatten()
        return x_flat.astype(complex)


# =====================================================================
# 3. Calibration and Loss Model
# =====================================================================
class CalibrationAndLoss:
    """Simulates physical hardware imperfections like optical insertion loss and phase drift."""

    def __init__(self, insertion_loss_db: float = 0.5, phase_noise_std: float = 0.01):
        self.attenuation = 10 ** (-abs(insertion_loss_db) / 20.0)
        self.phase_noise_std = phase_noise_std

    def apply_loss_and_calibration(self, signal: np.ndarray) -> np.ndarray:
        noise = np.random.normal(0, self.phase_noise_std, size=signal.shape)
        noisy_signal = signal * np.exp(1j * noise)
        return noisy_signal * self.attenuation


# =====================================================================
# 4. Signed Readout, Noise, and ReLU
# =====================================================================
class PhotonicReadout:
    """
    Handles photodetector measurement, noise, signed decoding, and non-linearities.

    FIX: the original readout returned sign(Re(signal)) * |signal|^2 —
    optical *power* detection. That's a quadratic function of the
    underlying linear value, so it can never match a linear baseline
    like Re(W @ x) even with a perfect encoder (e.g. a true value of 1.7
    would read out as ~2.89). For validation against a linear MVM
    baseline, decode the real part of the field directly instead of
    detector power.
    """

    def __init__(self, detector_noise_std: float = 0.005):
        self.detector_noise_std = detector_noise_std

    def measure_and_decode(self, optical_signal: np.ndarray) -> np.ndarray:
        """Linear homodyne-style readout: recovers Re(signal) directly."""
        readout = np.real(optical_signal)
        noise = np.random.normal(0, self.detector_noise_std, size=readout.shape)
        return readout + noise

    @staticmethod
    def relu(x: np.ndarray) -> np.ndarray:
        """Optical or post-processing non-linear activation."""
        return np.maximum(0, x)


# =====================================================================
# Full Integration Pipeline Wrapper
# =====================================================================
class MZIIntegrationPipeline:
    """
    Sequence: Input Vector/Patch -> MZI Matrix Calc -> Loss & Calibration -> Signed Readout -> Output
    """
    def __init__(self, loss_model: CalibrationAndLoss, readout_model: PhotonicReadout):
        self.loss_model = loss_model
        self.readout = readout_model

    def run(self, W: np.ndarray, x: np.ndarray) -> np.ndarray:
        x_encoded = InputEncoder.encode_vector(x)
        optical_out = MZICore.simulate_matrix_vector_mult(W, x_encoded)
        attenuated_out = self.loss_model.apply_loss_and_calibration(optical_out)
        readout_out = self.readout.measure_and_decode(attenuated_out)
        activated_out = self.readout.relu(readout_out)
        return activated_out


# =====================================================================
# 5. Workloads (Graphics, Convolution, AI, Scientific Computing)
# =====================================================================
class Workloads:
    @staticmethod
    def graphics_3d_transform(pipeline: MZIIntegrationPipeline, points: np.ndarray, transform_matrix: np.ndarray) -> np.ndarray:
        results = [pipeline.run(transform_matrix, pt) for pt in points]
        return np.array(results)

    @staticmethod
    def photonic_conv2d(pipeline: MZIIntegrationPipeline, image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
        i_h, i_w = image.shape
        k_h, k_w = kernel.shape
        out_h, out_w = i_h - k_h + 1, i_w - k_w + 1
        output = np.zeros((out_h, out_w))
        kernel_flat = kernel.flatten().reshape(1, -1)

        for r in range(out_h):
            for c in range(out_w):
                patch = image[r:r+k_h, c:c+k_w].flatten()
                output[r, c] = pipeline.run(kernel_flat, patch)[0]
        return output

    @staticmethod
    def ai_dense_layer(pipeline: MZIIntegrationPipeline, inputs: np.ndarray, weights: np.ndarray) -> np.ndarray:
        return pipeline.run(weights, inputs)

    @staticmethod
    def scientific_matrix_op(pipeline: MZIIntegrationPipeline, A: np.ndarray, b: np.ndarray) -> np.ndarray:
        """Scientific-computing MVM, e.g. one step of A @ b for a discretized
        operator (stiffness/diffusion-style matrix) acting on a state vector."""
        return pipeline.run(A, b)


# =====================================================================
# 6. Validation Tests Comparing Each Result with NumPy
# =====================================================================
def run_validation_tests(pipeline: MZIIntegrationPipeline):
    print("\n--- Running Validation Tests (MZI Pipeline vs NumPy Baseline) ---")

    np.random.seed(42)
    W = np.random.randn(4, 4)
    x = np.random.randn(4)

    numpy_ideal = np.maximum(0, np.real(W @ x))
    mzi_result = pipeline.run(W, x)

    mse = np.mean((numpy_ideal - mzi_result) ** 2)
    print(f"NumPy Reference Output : {np.round(numpy_ideal, 4)}")
    print(f"Photonic MZI Output    : {np.round(mzi_result, 4)}")
    print(f"Mean Squared Error (MSE): {mse:.6f}")


def run_scientific_computing_validation(pipeline: MZIIntegrationPipeline):
    """Validates a scientific-computing workload (a symmetric operator matrix
    applied to a state vector, e.g. one step of a discretized diffusion
    operator) against its own NumPy baseline, reported separately."""
    print("\n--- Running Scientific Computing Validation (Operator MVM vs NumPy Baseline) ---")

    np.random.seed(13)
    A_raw = np.random.randn(5, 5)
    A = (A_raw + A_raw.T) / 2  # symmetric, PDE-operator-like matrix
    np.random.seed(1006)  # chosen so ReLU doesn't zero out most components,
    b = np.random.randn(5)  # giving a non-degenerate MSE comparison

    numpy_ideal = np.maximum(0, np.real(A @ b))
    mzi_result = Workloads.scientific_matrix_op(pipeline, A, b)

    mse = np.mean((numpy_ideal - mzi_result) ** 2)
    print(f"NumPy Reference (sci. op) : {np.round(numpy_ideal, 4)}")
    print(f"Photonic MZI (sci. op)    : {np.round(mzi_result, 4)}")
    print(f"MSE (sci. op)             : {mse:.6f}")


def run_ai_workload_validation(pipeline: MZIIntegrationPipeline):
    """Validates the AI dense-layer workload against its own NumPy baseline,
    reported separately from the core MVM test above."""
    print("\n--- Running AI Workload Validation (Dense Layer vs NumPy Baseline) ---")

    np.random.seed(7)
    W_ai = np.random.randn(3, 3)
    x_ai = np.array([0.5, -0.2, 0.8])

    numpy_ideal = np.maximum(0, np.real(W_ai @ x_ai))
    mzi_result = Workloads.ai_dense_layer(pipeline, x_ai, W_ai)

    mse = np.mean((numpy_ideal - mzi_result) ** 2)
    print(f"NumPy Reference (AI layer): {np.round(numpy_ideal, 4)}")
    print(f"Photonic MZI (AI layer)   : {np.round(mzi_result, 4)}")
    print(f"MSE (AI layer)            : {mse:.6f}")


# =====================================================================
# 7. Status Report
# =====================================================================
def generate_status_report():
    report = """
    ====================================================================
                     MZI PROTOTYPE SYSTEM REPORT (FIXED)
    ====================================================================
    [STATUS] Hardware vs. Simulation Matrix

    Component                          | Status      | Implementation
    --------------------------------------------------------------------
    1. Core MZI Engine                 | SIMULATED   | SVD optical mesh equivalent
    2. Input Encoding                  | SIMULATED   | Amplitude-only, zero-phase
    3. Calibration & Optical Loss      | SIMULATED   | Attenuation & phase noise
    4. Signed Readout & ReLU           | SIMULATED   | Linear (Re) homodyne readout
    5. Workloads (Graphics/AI/Conv/Sci)| INTEGRATED  | Uses shared matrix math engine
    6. NumPy Validation Suite          | FUNCTIONAL  | MSE verified for MVM, AI layer, sci. op

    --------------------------------------------------------------------
    CHANGED FROM PREVIOUS VERSION:
    - Input encoding no longer applies a data-dependent phase or rescaling;
      it now matches the linear NumPy baseline by construction.
    - Readout now recovers Re(signal) instead of signed detector power
      (|signal|^2), so it stays linear like the baseline.
    - Residual MSE should now reflect only injected phase/detector noise
      and optical attenuation, not a structural mismatch.

    WHAT IS STILL SIMULATED:
    - Thermal phase shifter physics and physical waveguide geometry.
    - Physical hardware DAC/ADC quantization effects.

    NOTE: if you actually want to keep the physically-realistic
    quadratic power-detection readout (closer to what a real
    photodetector does), the correct fix is different: build the
    NumPy baseline as sign(Re(W@x)) * |W@x|^2 instead of making the
    readout linear. This version assumes you want the pipeline to
    validate against a plain linear MVM baseline.
    ====================================================================
    """
    print(report)


# =====================================================================
# Execution Script
# =====================================================================
if __name__ == "__main__":
    loss_model = CalibrationAndLoss(insertion_loss_db=0.2, phase_noise_std=0.005)
    readout = PhotonicReadout(detector_noise_std=0.001)
    pipeline = MZIIntegrationPipeline(loss_model, readout)

    run_validation_tests(pipeline)
    run_ai_workload_validation(pipeline)
    run_scientific_computing_validation(pipeline)

    generate_status_report()


--- Running Validation Tests (MZI Pipeline vs NumPy Baseline) ---
NumPy Reference Output : [0.     0.     1.7245 1.5141]
Photonic MZI Output    : [0.     0.     1.6841 1.48  ]
Mean Squared Error (MSE): 0.000699

--- Running AI Workload Validation (Dense Layer vs NumPy Baseline) ---
NumPy Reference (AI layer): [0.9647 0.3632 1.1646]
Photonic MZI (AI layer)   : [0.9432 0.3547 1.1379]
MSE (AI layer)            : 0.000416

--- Running Scientific Computing Validation (Operator MVM vs NumPy Baseline) ---
NumPy Reference (sci. op) : [1.0003 0.6128 1.7985 0.7798 0.3881]
Photonic MZI (sci. op)    : [0.9771 0.5995 1.7572 0.7605 0.3786]
MSE (sci. op)             : 0.000577

                     MZI PROTOTYPE SYSTEM REPORT (FIXED)
    [STATUS] Hardware vs. Simulation Matrix

    Component                          | Status      | Implementation
    --------------------------------------------------------------------
    1. Core MZI Engine                 | SIMULATED   | SVD optical mesh equivalent